# SER via YAMNet Transfer Learning

**Pipeline:**
1. Download RAVDESS + CREMA-D
2. Remap to 6 classes (Angry, Fear, Happy, Neutral, Sad, Surprise) — Disgust and Calm are dropped
3. Speaker-independent train/val/test split (80/10/10)
4. Extract YAMNet embeddings (1024-d per clip, mean-pooled over time)
5. Train small classifier head on embeddings
6. Save end-to-end model (raw audio in, 6 emotion probs out)

**Target:** val accuracy ≥ 75%, generalization to unseen speakers.

**Output artifacts saved to Google Drive:**
- `ser_yamnet_model.keras` — end-to-end model
- `ser_yamnet_classes.json` — class order
- `ser_yamnet_report.txt` — evaluation report

## Step 0 — Mount Drive and check GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = '/content/drive/MyDrive/BitirmeProjesi/ser_yamnet'
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output dir: {OUTPUT_DIR}')

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f'GPU: {gpus}')

## Step 1 — Install deps and download datasets

RAVDESS: 1440 clips, 24 actors.  
CREMA-D: 7442 clips, 91 actors (more diverse — this is what breaks the RAVDESS-only overfit).

In [ ]:
!pip install -q tensorflow_hub librosa soundfile tqdm

In [ ]:
# RAVDESS from Zenodo
!wget -q -O /content/RAVDESS.zip https://zenodo.org/records/1188976/files/Audio_Speech_Actors_01-24.zip
!mkdir -p /content/RAVDESS && unzip -q /content/RAVDESS.zip -d /content/RAVDESS
print('RAVDESS ready')

In [ ]:
# CREMA-D from GitHub. Takes ~5 minutes.
!git clone --depth 1 https://github.com/CheyneyComputerScience/CREMA-D.git /content/CREMA-D
print('CREMA-D ready')
!ls /content/CREMA-D/AudioWAV | head -5
!echo 'Total CREMA-D wavs:'
!ls /content/CREMA-D/AudioWAV | wc -l

## Step 2 — Build a unified index of clips

We want a single list: `(file_path, emotion_label, speaker_id)`.  
The speaker_id is critical: we'll split train/val/test **by speaker**, not by clip. This prevents the model from memorizing individual voices (the main failure mode of the old SER model).

**Label mapping (6 final classes):**
- 0: Angry
- 1: Fear
- 2: Happy
- 3: Neutral
- 4: Sad
- 5: Surprise

Disgust and Calm are dropped.

In [ ]:
import os
import pandas as pd

CLASSES = ['Angry', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

# RAVDESS filename: 03-01-EE-II-SS-RR-AA.wav
# EE (emotion): 01=neutral, 02=calm, 03=happy, 04=sad, 05=angry, 06=fearful, 07=disgust, 08=surprised
# AA (actor): 01-24
RAVDESS_EMOTION_MAP = {
    '01': 'Neutral',
    '03': 'Happy',
    '04': 'Sad',
    '05': 'Angry',
    '06': 'Fear',
    '08': 'Surprise',
    # '02' (calm) and '07' (disgust) are intentionally dropped
}

# CREMA-D filename: ActorID_Sentence_Emotion_Level.wav  e.g. 1001_DFA_ANG_XX.wav
CREMA_EMOTION_MAP = {
    'NEU': 'Neutral',
    'HAP': 'Happy',
    'SAD': 'Sad',
    'ANG': 'Angry',
    'FEA': 'Fear',
    # CREMA-D has no Surprise — only 5 emotions overlap. We'll rely on RAVDESS for Surprise.
    # 'DIS' (disgust) is dropped.
}

rows = []

# Index RAVDESS
ravdess_root = '/content/RAVDESS'
for actor_dir in sorted(os.listdir(ravdess_root)):
    actor_path = os.path.join(ravdess_root, actor_dir)
    if not os.path.isdir(actor_path):
        continue
    for fn in sorted(os.listdir(actor_path)):
        if not fn.endswith('.wav'):
            continue
        parts = fn.replace('.wav', '').split('-')
        emo_code = parts[2]
        actor = parts[6]
        if emo_code not in RAVDESS_EMOTION_MAP:
            continue
        rows.append({
            'path': os.path.join(actor_path, fn),
            'emotion': RAVDESS_EMOTION_MAP[emo_code],
            'speaker': f'RAV_{actor}',
            'source': 'RAVDESS',
        })

# Index CREMA-D
crema_root = '/content/CREMA-D/AudioWAV'
for fn in sorted(os.listdir(crema_root)):
    if not fn.endswith('.wav'):
        continue
    parts = fn.replace('.wav', '').split('_')
    if len(parts) < 3:
        continue
    actor = parts[0]
    emo_code = parts[2]
    if emo_code not in CREMA_EMOTION_MAP:
        continue
    rows.append({
        'path': os.path.join(crema_root, fn),
        'emotion': CREMA_EMOTION_MAP[emo_code],
        'speaker': f'CRE_{actor}',
        'source': 'CREMA-D',
    })

df = pd.DataFrame(rows)
df['label'] = df['emotion'].map(CLASS_TO_IDX)

print(f'Total clips: {len(df)}')
print(f'Unique speakers: {df["speaker"].nunique()}')
print()
print('Per-class counts:')
print(df['emotion'].value_counts())
print()
print('Per-source counts:')
print(df['source'].value_counts())

## Step 3 — Speaker-independent split

Shuffle the **speakers** (not the clips), then take the first 80% of speakers for train, next 10% for val, last 10% for test. This way the same voice never appears in two splits.

In [ ]:
import numpy as np

SEED = 42
rng = np.random.default_rng(SEED)

speakers = sorted(df['speaker'].unique())
rng.shuffle(speakers)
n = len(speakers)
n_train = int(0.8 * n)
n_val = int(0.1 * n)

train_speakers = set(speakers[:n_train])
val_speakers = set(speakers[n_train:n_train + n_val])
test_speakers = set(speakers[n_train + n_val:])

df['split'] = df['speaker'].apply(
    lambda s: 'train' if s in train_speakers else ('val' if s in val_speakers else 'test')
)

print(df.groupby(['split', 'emotion']).size().unstack(fill_value=0))
print()
print(f'Train clips: {(df["split"]=="train").sum()} / {len(train_speakers)} speakers')
print(f'Val   clips: {(df["split"]=="val").sum()} / {len(val_speakers)} speakers')
print(f'Test  clips: {(df["split"]=="test").sum()} / {len(test_speakers)} speakers')

## Step 4 — Load YAMNet and build an embedding extractor

YAMNet expects mono 16 kHz float32 audio in range [-1, 1].  
It outputs `(scores, embeddings, spectrogram)`. We use `embeddings`: shape `(T, 1024)` where T ≈ num_seconds * 2.  
We mean-pool across time → single 1024-d vector per clip.

In [ ]:
import tensorflow_hub as hub
import librosa
from tqdm.auto import tqdm

YAMNET_HANDLE = 'https://tfhub.dev/google/yamnet/1'
yamnet = hub.load(YAMNET_HANDLE)
print('YAMNet loaded')

TARGET_SR = 16000

def wav_to_embedding(path):
    y, _ = librosa.load(path, sr=TARGET_SR, mono=True)
    # Trim leading/trailing silence
    y, _ = librosa.effects.trim(y, top_db=30)
    if len(y) < TARGET_SR * 0.5:
        # Pad very short clips to 0.5s so YAMNet gets at least one frame
        y = np.pad(y, (0, TARGET_SR // 2 - len(y)))
    y = y.astype(np.float32)
    _, embeddings, _ = yamnet(y)
    return tf.reduce_mean(embeddings, axis=0).numpy()  # (1024,)

# Smoke test
sample_emb = wav_to_embedding(df.iloc[0]['path'])
print(f'Embedding shape: {sample_emb.shape}')

## Step 5 — Extract embeddings for all clips

This takes ~10-15 minutes on Colab GPU for ~8800 clips. Cached to Drive so you don't re-run it.

In [ ]:
CACHE_PATH = os.path.join(OUTPUT_DIR, 'embeddings_cache.npz')

if os.path.exists(CACHE_PATH):
    print(f'Loading cached embeddings from {CACHE_PATH}')
    cache = np.load(CACHE_PATH)
    X_all = cache['X']
    y_all = cache['y']
    splits = cache['splits']
    print(f'Cached X shape: {X_all.shape}')
else:
    print(f'Extracting embeddings for {len(df)} clips...')
    X_list, y_list, split_list = [], [], []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        try:
            emb = wav_to_embedding(row['path'])
            X_list.append(emb)
            y_list.append(row['label'])
            split_list.append(row['split'])
        except Exception as e:
            print(f'Skip {row["path"]}: {e}')
    X_all = np.stack(X_list).astype(np.float32)
    y_all = np.array(y_list, dtype=np.int32)
    splits = np.array(split_list)
    np.savez_compressed(CACHE_PATH, X=X_all, y=y_all, splits=splits)
    print(f'Saved cache to {CACHE_PATH}')

train_mask = splits == 'train'
val_mask = splits == 'val'
test_mask = splits == 'test'

X_train, y_train = X_all[train_mask], y_all[train_mask]
X_val, y_val = X_all[val_mask], y_all[val_mask]
X_test, y_test = X_all[test_mask], y_all[test_mask]

print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')

## Step 6 — Train the classifier head

Input: 1024-d YAMNet embedding.  
Output: 6-class softmax.  
Class weights added because Surprise is underrepresented (only in RAVDESS).

In [ ]:
from tensorflow.keras import layers, models, callbacks
from sklearn.utils.class_weight import compute_class_weight

NUM_CLASSES = len(CLASSES)
EMB_DIM = X_train.shape[1]

cw = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=y_train)
class_weight = {i: float(w) for i, w in enumerate(cw)}
print(f'Class weights: {class_weight}')

def build_head(input_dim, num_classes):
    m = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax'),
    ])
    m.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return m

head = build_head(EMB_DIM, NUM_CLASSES)
head.summary()

In [ ]:
early = callbacks.EarlyStopping(
    monitor='val_accuracy', patience=15, restore_best_weights=True, mode='max'
)
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=5, min_lr=1e-5
)

history = head.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=64,
    class_weight=class_weight,
    callbacks=[early, reduce_lr],
    verbose=2,
)

## Step 7 — Evaluate on held-out test speakers

In [ ]:
import matplotlib.pyplot as plt

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.plot(history.history['accuracy'], label='train')
a1.plot(history.history['val_accuracy'], label='val')
a1.set_title('Accuracy'); a1.legend(); a1.grid(True)
a2.plot(history.history['loss'], label='train')
a2.plot(history.history['val_loss'], label='val')
a2.set_title('Loss'); a2.legend(); a2.grid(True)
plt.show()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

val_loss, val_acc = head.evaluate(X_val, y_val, verbose=0)
test_loss, test_acc = head.evaluate(X_test, y_test, verbose=0)
print(f'Val  accuracy: {val_acc:.4f}')
print(f'Test accuracy: {test_acc:.4f}')

y_pred = head.predict(X_test, verbose=0).argmax(axis=1)
report = classification_report(y_test, y_pred, target_names=CLASSES, digits=3)
print(report)

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion Matrix (Test)')
plt.show()

## Step 8 — Build end-to-end model (raw audio → emotion)

We wrap YAMNet + classifier head in a single `tf.keras.Model`. The saved file is self-contained: `main.py` just loads it and passes in a 16 kHz mono waveform.

In [ ]:
class E2EModel(tf.keras.Model):
    def __init__(self, yamnet_handle, head_model, **kwargs):
        super().__init__(**kwargs)
        self.yamnet = hub.KerasLayer(yamnet_handle, trainable=False)
        self.head = head_model

    def call(self, waveform, training=False):
        # waveform: 1-D float32 tensor, 16 kHz
        _, embeddings, _ = self.yamnet(waveform)
        pooled = tf.reduce_mean(embeddings, axis=0, keepdims=True)
        return self.head(pooled, training=training)

e2e = E2EModel(YAMNET_HANDLE, head)
# Warm up with dummy input
dummy = tf.zeros([TARGET_SR], dtype=tf.float32)
_ = e2e(dummy)
print('E2E ready')

In [ ]:
# Save via SavedModel format — works reliably with hub.KerasLayer
E2E_SAVED_DIR = os.path.join(OUTPUT_DIR, 'ser_yamnet_saved_model')
tf.saved_model.save(e2e, E2E_SAVED_DIR)
print(f'SavedModel: {E2E_SAVED_DIR}')

# Also save head-only as .keras for flexibility
HEAD_PATH = os.path.join(OUTPUT_DIR, 'ser_yamnet_head.keras')
head.save(HEAD_PATH)
print(f'Head: {HEAD_PATH}')

# Save class order + metadata
import json
meta = {
    'classes': CLASSES,
    'target_sample_rate': TARGET_SR,
    'val_accuracy': float(val_acc),
    'test_accuracy': float(test_acc),
    'num_train': int(len(y_train)),
    'num_val': int(len(y_val)),
    'num_test': int(len(y_test)),
    'sources': ['RAVDESS', 'CREMA-D'],
    'architecture': 'YAMNet embeddings (1024-d) -> Dense(256) -> Dense(128) -> Softmax(6)',
}
with open(os.path.join(OUTPUT_DIR, 'ser_yamnet_meta.json'), 'w') as f:
    json.dump(meta, f, indent=2)

with open(os.path.join(OUTPUT_DIR, 'ser_yamnet_report.txt'), 'w') as f:
    f.write(report)

print('All artifacts saved to:', OUTPUT_DIR)

## Step 9 — Live test on your own voice

Record in the browser, check what the model says. This is the **real** test — speaker-independent test accuracy is one number, but hearing the model work on your actual voice is what tells you if it'll work in `main.py`.

In [ ]:
# Browser mic recording widget
from IPython.display import Javascript, display
from google.colab import output as colab_output
import base64

RECORD = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time));
const b2t = blob => new Promise(r => { const f = new FileReader(); f.onloadend = e => r(e.target.result); f.readAsDataURL(blob); });
var record = time => new Promise(async resolve => {
  const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
  const recorder = new MediaRecorder(stream);
  const chunks = [];
  recorder.ondataavailable = e => chunks.push(e.data);
  recorder.start();
  await sleep(time);
  recorder.stop();
  await new Promise(r => recorder.onstop = r);
  const blob = new Blob(chunks);
  const b64 = await b2t(blob);
  resolve(b64);
});
"""

def record_audio(seconds=4):
    display(Javascript(RECORD))
    b64 = colab_output.eval_js(f'record({seconds*1000})')
    data = base64.b64decode(b64.split(',')[1])
    with open('/content/mic.webm', 'wb') as f:
        f.write(data)
    !ffmpeg -y -i /content/mic.webm -ar 16000 -ac 1 /content/mic.wav -loglevel error
    return '/content/mic.wav'

In [ ]:
print('Speak for 4 seconds...')
path = record_audio(4)
y, _ = librosa.load(path, sr=TARGET_SR, mono=True)
y, _ = librosa.effects.trim(y, top_db=30)
probs = e2e(tf.constant(y, dtype=tf.float32)).numpy()[0]

print(f'Trimmed duration: {len(y)/TARGET_SR:.2f}s')
for c, p in sorted(zip(CLASSES, probs), key=lambda x: -x[1]):
    bar = '█' * int(p * 40)
    print(f'{c:10s} {p:.3f} {bar}')

## What to check

1. **Test accuracy ≥ 70%** — speaker-independent, so this is the honest number.
2. **No single class collapse in the confusion matrix** — old model collapsed to Neutral/Fearful; new one should spread its predictions.
3. **Live test** — try 4-5 different emotions on your own voice. It should get the clear ones right (angry shout, happy laugh, sad sigh).

If live test looks off on borderline cases, that's fine — fusion with vision will pick up the slack. If it's totally broken (everything → Neutral), something's wrong, ping me.